# Extractive QA with ranked, confidence-scored answers — end-to-end walkthrough**Ref:** NX-NLP-2026-018 · Verbalis AI technical assessment**Runtime:** Colab (T4 GPU) or a local Jupyter kernel · **Wall-clock:** ~25 min at the default sample sizeThe production logic lives in `src/qa_system/` and is imported here. The notebook neverre-implements it, so what runs in this walkthrough is exactly what the Flask service serves.| Requirement | Where it is satisfied ||---|---|| 4.1 Same tokenisation for question and context | `qa_system.preprocessing` — §5 || 4.2 Contextual embeddings from a pretrained transformer | `qa_system.modeling` — §6 || 4.3 HF inference pipeline out of the box **+** fine-tuning | `qa_system.baseline` — §4; `qa_system.train` — §7 || 4.4 Span extraction, confidence score, ranked order | `qa_system.postprocessing` — §8 || 4.5 Save / reload, user-set number of candidates | `qa_system.modeling`, `QAConfig.top_k` — §10, §11 || 4.6 Flask web application, top-k per query | `app/flask_app.py` — §13 || 5. Input: question + one or more passages · Output: ranked answers with scores | `qa_system.inference` — §11 |

## 1. Design decisions (the short version)| Decision | Choice | Why ||---|---|---|| Task framing | **Extractive span prediction** | The brief requires the answer to be drawn from the context. A span head cannot hallucinate — every output is a substring of an input passage. || Confidence score | `P(start=i) · P(end=j)` from a **masked softmax**, not `start_logit + end_logit` | Logit sums rank correctly *within* one forward pass but are unbounded and un-normalised, so they cannot be compared across passages. This system merges candidates from several passages into one list, so the score must live in [0,1] and mean the same thing everywhere. It is also the convention the HF pipeline uses, which makes the baseline and the fine-tuned model directly comparable. || Multi-passage merge | Pool all candidates, then **aggregate duplicates** | The same answer found in two independent passages is stronger evidence than one slightly more confident lone mention. Switchable via `cfg.aggregate_duplicates`. || Baseline | `distilbert-base-cased-distilled-squad` through `pipeline()` | Requirement 4.3 asks for out-of-the-box inference; it also gives an honest reference point — a fine-tune that cannot beat it is not worth shipping. It means the Flask app answers before any training has happened. || Fine-tuning data | **SQuAD v2.0** | v2 adds unanswerable questions, so the model learns to put mass on `[CLS]` instead of forcing a span. That is what stops a confident wrong answer topping the ranking on an irrelevant passage. || Long passages | Sliding window, `max_seq_length=384`, `stride=128` | Criterion 6.2 grades long and noisy contexts. Truncation silently deletes answers; the 128-token overlap stops an answer being split across a boundary. || Metrics | EM / F1 **and** Hit@k, Recall@k, MRR | Criterion 6.1 grades whether *the most likely answer appears first*. EM measures only rank 1, so it cannot tell a model that ranks well from one that buries the right answer at position 5. || Training loop | Native PyTorch, not `Trainer` | `TrainingArguments` keyword names have churned across releases; an explicit loop is version-stable and makes warm-up, clipping and AMP reviewable. |

## 2. Environment

In [ ]:
# !pip install -q "transformers>=4.38" "datasets>=2.18" "torch>=2.1" flask

In [ ]:
import os, sys, json, textwrapfrom functools import partialimport numpy as npimport torchPROJECT_ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))from qa_system.config import QAConfigfrom qa_system.data import load_qa_datasets, dataset_summaryfrom qa_system.preprocessing import (    prepare_train_features, prepare_validation_features, featurise_single, describe_windowing,)from qa_system.modeling import (    build_model_and_tokenizer, save_artifacts, load_artifacts, count_parameters, get_device,)from qa_system.baseline import build_qa_pipeline, baseline_answerfrom qa_system.train import train_model, set_seedfrom qa_system.postprocessing import extract_candidates, rank_candidates, answer_question_over_passagesfrom qa_system.metrics import (    squad_metrics, ranking_metrics, confidence_calibration, normalize_answer,)from qa_system.evaluation import evaluate_model, collect_logits, references_from_examplesfrom qa_system.inference import RankedQAPipelinedevice = get_device()print("torch", torch.__version__, "| device:", device)

## 3. ConfigurationOne dataclass drives training, evaluation and serving, and it is saved next to the weights — so acheckpoint carries its own window size, stride and default `top_k`.`max_train_samples` is capped so the whole notebook runs in a demo session. Set both caps to `None`for a full SQuAD v2 run (~130k examples, ~1.5 h/epoch on a T4).

In [ ]:
cfg = QAConfig(    model_name="distilbert-base-uncased",    baseline_model_name="distilbert-base-cased-distilled-squad",    dataset_name="squad_v2",    max_train_samples=8000,      # None -> full 130k train set    max_eval_samples=1500,       # None -> full 11.8k dev set    max_seq_length=384,    doc_stride=128,    top_k=5,    num_train_epochs=2,    output_dir=os.path.join(PROJECT_ROOT, "artifacts", "qa-model"),)set_seed(cfg.seed)print(json.dumps(cfg.__dict__, indent=2))

## 4. Out-of-the-box inference (Requirement 4.3, first half)Before any training: a Hugging Face `question-answering` pipeline, a checkpoint already fine-tunedon SQuAD, and `top_k` ranked candidates back. This is the baseline every later number is measuredagainst, and it is what the Flask app falls back to when no fine-tuned checkpoint exists yet.`qa_system.baseline` wraps the pipeline to add the two things the brief needs that the stock helperdoes not do on its own: **several passages in one call**, and **duplicate aggregation** across them.

In [ ]:
qa_pipe = build_qa_pipeline(cfg.baseline_model_name)manual = [    "The XR-500 router supports dual-band Wi-Fi on 2.4 GHz and 5 GHz. Firmware updates are "    "delivered automatically every Tuesday at 03:00 local time and take about four minutes.",    "The XR-500 carries a two-year limited warranty covering manufacturing defects. The warranty "    "is void if the casing is opened. Accessories are covered for ninety days.",]for cand in baseline_answer(qa_pipe, "How long is the warranty?", manual, top_k=5):    print(f"  {cand['rank']}. {cand['text']:<40} {cand['confidence']:.4f}  passage {cand['passage_index']}")

In [ ]:
# Ranking behaviour is the point of the exercise: a question whose answer appears# in more than one passage should surface it above a single stronger mention.sources = [    "Founded in 2019, the company opened its head office in Bengaluru and a sales office in Pune.",    "All corporate functions report to the head office in Bengaluru, which moved to Whitefield in 2024.",    "The support centre operates from Hyderabad on weekdays.",]for cand in baseline_answer(qa_pipe, "Where is the head office?", sources, top_k=4):    supp = f" (in {cand['support']} passages)" if cand["support"] > 1 else ""    print(f"  {cand['rank']}. {cand['text']:<25} {cand['confidence']:.4f}{supp}")

## 5. Data and preprocessing (Requirements 4.1, and the labels for 4.3)SQuAD v2.0 from the Hub. To run on a private corpus, drop two SQuAD-format JSON files in `data/`and set `dataset_name="local"` — `data/train_sample.json` shows the schema.

In [ ]:
raw = load_qa_datasets(cfg)print(raw)print(json.dumps(dataset_summary(raw), indent=2))ex = raw["train"][0]print("\nQUESTION:", ex["question"])print("ANSWERS :", ex["answers"])print("CONTEXT :", textwrap.fill(ex["context"][:400], 100))

Two things happen in preprocessing, and both are where extractive QA systems usually break.**(a) One tokenisation for both inputs.** The question and the context are tokenised *as a pair*by the same Hugging Face fast tokenizer, with `truncation="only_second"` so the question is nevercut. Pairs longer than 384 tokens become several overlapping windows (`stride=128`).**(b) Character → token alignment.** The dataset labels answers as character offsets; the modelpredicts token indices. `return_offsets_mapping=True` gives the character span of each token, whichmakes the conversion exact. A window that does not fully contain the answer — and any unanswerablequestion — is labelled at `[CLS]`, and only tokens with `sequence_id == 1` can ever be part of ananswer.

In [ ]:
model, tokenizer = build_model_and_tokenizer(cfg)print(type(tokenizer).__name__, "| fast:", tokenizer.is_fast)print("parameters:", count_parameters(model))# (a) the sliding window, made visiblelong_context = ("Retrieval systems index documents. " * 120) + \               " The service level agreement guarantees a response within four hours."info = describe_windowing("What does the SLA guarantee?", long_context, tokenizer, cfg)print(f"\ncontext chars: {info['n_context_chars']}  ->  {info['n_windows']} windows")for w in info["windows"]:    print(f"  window {w['window']}: tokens={w['n_tokens']:<4} chars {w['char_start']}-{w['char_end']}")print("Consecutive windows overlap, so an answer on a boundary is still whole in one of them.")

In [ ]:
# (b) verify the alignment is lossless before training on itsample = raw["train"].select(range(200))feats = prepare_train_features(sample.to_dict(), tokenizer, cfg)gold_pool = {normalize_answer(t) for a in sample["answers"] for t in a["text"]}checked = mismatches = 0for i, (s, e) in enumerate(zip(feats["start_positions"], feats["end_positions"])):    if s == 0 and e == 0:        continue                       # CLS = 'no answer in this window'    checked += 1    decoded = tokenizer.decode(feats["input_ids"][i][s:e + 1])    if normalize_answer(decoded) not in gold_pool:        mismatches += 1        if mismatches <= 5:            print("  mismatch:", repr(decoded))print(f"labelled spans checked: {checked} | not matching any gold answer: {mismatches}")

In [ ]:
train_features = raw["train"].map(    partial(prepare_train_features, tokenizer=tokenizer, cfg=cfg),    batched=True, remove_columns=raw["train"].column_names, desc="tokenising train",)eval_features = raw["validation"].map(    partial(prepare_validation_features, tokenizer=tokenizer, cfg=cfg),    batched=True, remove_columns=raw["validation"].column_names, desc="tokenising validation",)print(f"train:      {len(raw['train']):>6} examples -> {len(train_features):>6} features")print(f"validation: {len(raw['validation']):>6} examples -> {len(eval_features):>6} features")print("features > examples because long passages produce several windows")

## 6. Model (Requirement 4.2)`AutoModelForQuestionAnswering` = pretrained encoder + one linear layer projecting each token'scontextual embedding to two logits: *is this token the start?* / *is this token the end?*Loss is cross-entropy over start positions plus cross-entropy over end positions. Because `[CLS]`is a legal target, "no answer" is learned by the same loss — no separate head.

In [ ]:
print(model.config.architectures, "| hidden size:", model.config.hidden_size)with torch.no_grad():    demo = tokenizer("Who wrote Hamlet?", "Hamlet is a tragedy written by William Shakespeare.",                     return_tensors="pt")    out = model(**demo)print("start_logits:", tuple(out.start_logits.shape), " end_logits:", tuple(out.end_logits.shape))print("-> one score per token per boundary; the head is untrained, so the argmax is meaningless yet.")

## 7. Fine-tuning (Requirement 4.3, second half)AdamW, 10% linear warm-up then linear decay, gradient clipping at 1.0, mixed precision on CUDA,no weight decay on biases or LayerNorm. Evaluation runs after every epoch, reporting both spanmetrics and ranking metrics, so a model that gets *worse at ranking* while EM improves is visibleimmediately.

In [ ]:
history = train_model(    model, tokenizer, train_features, cfg,    eval_examples=raw["validation"], eval_features=eval_features,)print(f"\ntrained in {history['train_seconds']}s")print(json.dumps(history["epoch_eval"], indent=2))

In [ ]:
import matplotlib.pyplot as pltsteps = [h["step"] for h in history["loss"]]fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))ax[0].plot(steps, [h["loss"] for h in history["loss"]]); ax[0].set_title("training loss")ax[1].plot(steps, [h["lr"] for h in history["loss"]]); ax[1].set_title("learning rate (warm-up + decay)")for a in ax: a.set_xlabel("optimiser step"); a.grid(alpha=.3)plt.tight_layout(); plt.show()

## 8. Scoring and ranking (Requirement 4.4)Per window: mask every position that cannot legally start or end an answer, softmax the start andend logits over what remains, then score each candidate span```confidence = P(start = i) · P(end = j)```Invalid pairs are dropped first — end before start, span longer than `max_answer_length`, or anendpoint outside the context. Surviving spans are mapped back to character offsets and sliced fromthe **original** string, so casing and punctuation survive an uncased tokenizer.Then candidates from every window and every passage are pooled and ranked globally.

In [ ]:
model.eval().to(device)def logits_for(question, context):    inputs, offsets = featurise_single(question, context, tokenizer, cfg)    with torch.no_grad():        out = model(**{k: v.to(device) for k, v in inputs.items()})    return offsets, out.start_logits.float().cpu().numpy(), out.end_logits.float().cpu().numpy()offsets, sl, el = logits_for("How long is the warranty?", manual[1])single = extract_candidates(manual[1], offsets, sl, el, cfg)print("candidates from ONE passage (probability vs raw logit sum):")for c in single["candidates"][:6]:    print(f"  {c['confidence']:.4f}   logit_sum={c['logit_score']:+.2f}   {c['text']!r}")print(f"\nP(no answer) for this passage: {single['null_prob']:.4f}")

In [ ]:
# Across all passages, with duplicate aggregation.offs, starts, ends = zip(*[logits_for("Where is the head office?", c) for c in sources])ranked = answer_question_over_passages(sources, offs, starts, ends, cfg, top_k=5)for c in ranked:    supp = f"  <- corroborated by {c['support']} passages" if c["support"] > 1 else ""    print(f"  {c['rank']}. {c['text']:<25} conf={c['confidence']:.4f} "          f"passages={c['passages']}{supp}")

In [ ]:
# Aggregation on vs off, on the same logits: this is a product decision, so make it visible.cfg_no_agg = QAConfig(**{**cfg.__dict__, "aggregate_duplicates": False})plain = answer_question_over_passages(sources, offs, starts, ends, cfg_no_agg, top_k=5)print("aggregate_duplicates=False:", [c["text"] for c in plain])print("aggregate_duplicates=True :", [c["text"] for c in ranked])

## 9. Evaluation**Span quality** — Exact Match and F1 on the rank-1 answer, split into answerable (`HasAns`) andunanswerable (`NoAns`). On SQuAD v2 the aggregate alone is misleading: abstaining on everythingscores about 50.**Ranking quality** — Hit@k (is a correct answer anywhere in the top k?), Recall@k (is a *useful*answer there, F1 ≥ 0.5?) and MRR (how high does the first correct answer sit?). These are whatcriterion 6.1 actually asks about. The gap between Hit@1 and Hit@5 is the headroom a re-rankerwould recover.**Calibration** — a 0.8-confidence answer should be right about 80% of the time. If accuracy doesnot rise with confidence, the number shown next to each answer in the UI is decoration.

In [ ]:
report = evaluate_model(model, raw["validation"], eval_features, cfg, device=device, k_values=(1, 3, 5))print("span quality (rank-1)")print(json.dumps(report["span_metrics"], indent=2))print("\nranking quality")print(json.dumps(report["ranking_metrics"], indent=2))

In [ ]:
import pandas as pdcal = pd.DataFrame(report["calibration"])display(cal)if len(cal):    plt.figure(figsize=(5, 3.4))    plt.bar(cal["confidence_bin"], cal["accuracy"], color="#17667e")    plt.ylabel("exact match (%)"); plt.xlabel("rank-1 confidence")    plt.title("Does confidence mean anything?"); plt.tight_layout(); plt.show()

In [ ]:
# Fine-tuned vs out-of-the-box, on the same questions. If the baseline wins, ship the baseline.val_ids = list(report["references"].keys())[:200]subset = raw["validation"].filter(lambda e: e["id"] in set(val_ids))base_ranked, refs = {}, {}for row in subset:    cands = baseline_answer(qa_pipe, row["question"], [row["context"]], top_k=5)    base_ranked[row["id"]] = [c["text"] for c in cands]    refs[row["id"]] = list(row["answers"]["text"])ours = {i: [c["text"] for c in report["ranked_predictions"][i] if not c["is_no_answer"]]        for i in refs}print("baseline  ", json.dumps(ranking_metrics(base_ranked, refs), indent=None))print("fine-tuned", json.dumps(ranking_metrics(ours, refs), indent=None))

In [ ]:
# Error analysis on the ranked list — where do the misses actually come from?rows = []for ex_id, golds in report["references"].items():    if not golds:        continue    cands = [c["text"] for c in report["ranked_predictions"][ex_id] if not c["is_no_answer"]]    em = [max(int(normalize_answer(g) == normalize_answer(c)) for g in golds) for c in cands]    first = next((i + 1 for i, v in enumerate(em) if v == 1), None)    rows.append({        "id": ex_id,        "outcome": "correct @1" if first == 1 else                   f"recoverable @{first}" if first else "not in top-k",        "gold": golds[0],        "top1": cands[0] if cands else "",    })err = pd.DataFrame(rows)display(err["outcome"].value_counts().to_frame("count"))display(err[err.outcome.str.startswith("recoverable")].head(8))

## 10. Save and reload (Requirement 4.5)Weights, tokenizer and `qa_config.json` go to one directory. Reloading restores identical decodingbehaviour — window size, stride, default `top_k`, aggregation setting — with no retraining.

In [ ]:
model_dir = save_artifacts(model, tokenizer, cfg)print("saved to:", model_dir)print(sorted(os.listdir(model_dir)))del modelreloaded, reloaded_tok, reloaded_cfg = load_artifacts(model_dir)print("\nreloaded top_k:", reloaded_cfg.top_k,      "| max_seq_length:", reloaded_cfg.max_seq_length,      "| stride:", reloaded_cfg.doc_stride,      "| aggregate:", reloaded_cfg.aggregate_duplicates)

## 11. The serving pipeline (Requirements 4.5, 5)`RankedQAPipeline` is the only object the Flask app imports. Input: a question and one or morepassages, plus how many candidates to return. Output: a ranked list, each entry carrying itsconfidence, its passage index and its character span, so the UI can highlight the evidence.

In [ ]:
qa = RankedQAPipeline.from_pretrained(model_dir)print(json.dumps(qa.info, indent=2))result = qa.answer("Where is the head office?", sources, top_k=3)print(json.dumps(result, indent=2)[:900], "...")

In [ ]:
# The user controls how many candidates come back (Requirement 4.5 / 4.6).for k in (1, 3, 5):    r = qa.answer("How long is the warranty?", manual, top_k=k)    print(f"top_k={k}: " + " | ".join(f"{c['text']} ({c['confidence']:.3f})" for c in r["answers"]))

## 12. Robustness and generalisation (Criteria 6.2, 6.3)

In [ ]:
# (a) long passage: the answer sits past the 384-token cut-offburied = ("Background information about the archive. " * 200) + \         " The retention period for audit logs is seven years."r = qa.answer("What is the retention period for audit logs?", buried, top_k=3)print(f"(a) long passage  | windows={r['n_windows']:<3} top: {r['answers'][0]['text']!r}")# (b) messy text: irregular whitespace, casing, stray markupmessy = ("invoice   NO. 44-B \n\n  the   PAYMENT is  due  within THIRTY (30) days   of receipt.\t\n"         "<b>note</b>: late fees apply.")r = qa.answer("When is the payment due?", messy, top_k=3)print(f"(b) messy text    | top: {r['answers'][0]['text']!r}")# (c) noisy distractors: the right passage sits among unrelated onesnoisy = [    "The cafeteria menu changes every Monday and vegetarian options are always available.",    "Parking permits cost 500 rupees per month and are issued by the facilities desk.",    "Employees may carry forward a maximum of fifteen unused leave days into the next year.",    "The gym on level two is open from 6 am to 10 pm on weekdays.",]r = qa.answer("How many leave days can be carried forward?", noisy, top_k=3)for c in r["answers"]:    print(f"(c) noisy set     | {c['rank']}. {c['text']!r} conf={c['confidence']:.3f} "          f"passage={c['passage_index']}")

In [ ]:
# (d) generalisation: hand-written questions over a document from neither splitheld_out = ("The Kaveri river rises in the Western Ghats of Karnataka and flows south-east across "            "Tamil Nadu before reaching the Bay of Bengal. Its basin covers roughly 81,000 square "            "kilometres and supports irrigation for both states, which led to a long-running "            "water-sharing dispute settled by a tribunal award in 2007.")for q in ["Where does the Kaveri river rise?",          "How large is the basin?",          "When was the dispute settled?",          "How many dams are on the river?"]:    r = qa.answer(q, held_out, top_k=2)    top = r["answers"][0]    print(f"{q:<38} -> {top['text']!r} ({top['confidence']:.3f})")

## 13. The Flask application (Requirement 4.6)The saved directory is all the service needs. If it does not exist yet, the app falls back to theout-of-the-box SQuAD checkpoint, so the endpoint is never dead.```bashexport QA_MODEL_DIR=artifacts/qa-modelpython app/flask_app.py            # http://127.0.0.1:5000``````bashcurl -s http://127.0.0.1:5000/api/answer \  -H 'Content-Type: application/json' \  -d '{"question": "How long is the warranty?",       "contexts": ["The XR-500 carries a two-year limited warranty.",                    "Accessories are covered for ninety days."],       "top_k": 3,       "engine": "finetuned"}'```The UI takes any number of passages, exposes `top_k` as a slider, lets you switch between thefine-tuned model and the stock pipeline, and highlights the supporting span in the source passagewhen a candidate is clicked.The HTTP layer imports torch lazily, so `tests/test_api.py` exercises the whole contract with a stubpipeline in milliseconds — no model, no GPU.## 14. What I would do next* **Retrieval front end.** The user supplies the passages today. Over a real knowledge base, add  BM25 or a dense retriever, take the top-k documents and let this reader rank across them — the  multi-passage path is already built for exactly that.* **Cross-encoder re-ranker.** The gap between Hit@1 and Hit@5 in §9 is recoverable headroom: a  re-ranker scoring (question, candidate, evidence sentence) typically converts a good part of it.* **Calibrate the confidence.** The softmax product ranks well but is not a probability of being  right. Fit an isotonic or Platt calibrator on validation so the number in the UI means something.* **Bigger backbone.** DeBERTa-v3-base or ELECTRA-large usually adds 6–10 F1 through the same code  path; DistilBERT was chosen for demo turnaround.* **Serving cost.** ONNX Runtime export with int8 quantisation is roughly a 3× CPU latency win for  a fraction of a point of F1.